# A2 Q2 - Train the Stage-2 re-ranker (run this notebook on Kaggle)

Trains one LightGBM re-ranker per dataset on the design matrix prepared
locally by `src/reranker_scores.ipynb`, and writes
`reranker_model_{dataset}.txt` + `reranker_metadata_{dataset}.json` to
`/kaggle/working/` for download.

**Upload as a Kaggle dataset:** one `reranker_training_{dataset}.parquet`
per dataset. Each carries the join keys, the `split` column, the `clicked`
label, and the 15 features listed in SPEC.md's `A2 Q2` section. The join is
done locally so only what training needs is uploaded (~350MB for
`ebnerd_large`, ~1.2GB for `mind_large`) rather than Q1's full 3.17GB
feature table.

**Why Kaggle only trains.** Stage-1 scoring needs the BM25 index and
embedding matrix, which live locally and take hours to apply; Kaggle hosts
the fitting step only, the same split used for the embeddings in A1 Q3.

**Learning curve first, final model second.** The training set is a
400,000-impression sample (SPEC.md A2 Q2 #3). Rather than assert that is
enough, this notebook fits the same configuration at 100k/200k/400k and
reports validation AUC for each, so the design note can state whether the
curve had flattened rather than defend a judgement call.

In [ ]:
import glob
import json
import os
import time

import lightgbm as lgb
import numpy as np
import pandas as pd

print("lightgbm", lgb.__version__)

# Must match cs4406m26_assignment1c1.reranker.FEATURE_COLUMNS exactly, in
# order: LightGBM identifies features positionally, so a booster trained
# here and served locally through a different order would score silently
# wrong rather than raise. The metadata written at the end records the
# order so the serving adapter can assert on it.
BEHAVIOURAL_FEATURES = [
    "click_count",
    "weighted_category_affinity",
    "weighted_read_time",
    "weighted_scroll_percentage",
    "weighted_embedding_similarity",
    "position_in_impression",
    "clicks_earlier_in_session",
    "session_impressions_so_far",
    "popularity",
    "freshness_hours",
    "category_match",
]
RETRIEVAL_FEATURES = ["bm25_score", "embedding_score", "in_bm25_top200", "in_embedding_top200"]
FEATURE_COLUMNS = BEHAVIOURAL_FEATURES + RETRIEVAL_FEATURES

LEARNING_CURVE_IMPRESSIONS = [100_000, 200_000, 400_000]
SEED = 0

# Pointwise binary objective as the default: the ranking metrics this is
# judged on (AUC/MRR/nDCG) are computed per impression afterwards either
# way, and lambdarank additionally needs per-impression group bookkeeping
# that has to stay consistent across two datasets with very different
# candidate counts (~11.9 vs ~37). lambdarank is the documented alternative,
# not the default (SPEC.md A2 Q2 #2).
PARAMS = {
    "objective": "binary",
    "metric": ["auc"],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": SEED,
    "num_threads": os.cpu_count(),
}
NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 50

## Locate the uploaded training matrices

In [ ]:
paths = sorted(glob.glob("/kaggle/input/**/reranker_training_*.parquet", recursive=True))
datasets = {os.path.basename(p)[len("reranker_training_"):-len(".parquet")]: p for p in paths}
assert datasets, f"no reranker_training_*.parquet under /kaggle/input -- found: {glob.glob('/kaggle/input/*')}"

frames = {}
for name, path in datasets.items():
    df = pd.read_parquet(path)
    missing = [c for c in FEATURE_COLUMNS + ["clicked", "split", "impression_id"] if c not in df.columns]
    assert not missing, f"{name}: missing columns {missing}"
    frames[name] = df
    n_impr = df["impression_id"].nunique()
    print(f"{name}: {len(df):,} rows, {n_impr:,} impressions, positive rate {df['clicked'].mean():.4f}")
    print(f"   splits: {df['split'].value_counts().to_dict()}")
    all_nan = [c for c in FEATURE_COLUMNS if df[c].isna().all()]
    print(f"   all-NaN features (absent for this dataset): {all_nan}")

In [ ]:
def test_loaded_matrices():
    for name, df in frames.items():
        assert set(df["split"].unique()) <= {"train", "val"}, df["split"].unique()
        assert df["clicked"].dtype == bool or set(df["clicked"].unique()) <= {0, 1}
        # every impression must sit entirely in one split, or the val score
        # would be measured on impressions the model partly trained on
        per_impr = df.groupby("impression_id")["split"].nunique()
        assert (per_impr == 1).all(), "an impression spans both splits"
        # both classes must be present, otherwise AUC is undefined
        for split in ("train", "val"):
            sub = df[df["split"] == split]
            assert sub["clicked"].nunique() == 2, f"{name}/{split}: single-class labels"
        # the two retrieval scores must be real numbers, not silently absent
        for col in ("bm25_score", "embedding_score"):
            assert df[col].notna().all(), f"{name}: {col} has nulls"


test_loaded_matrices()
print("ok: every impression sits in exactly one split, both classes present, retrieval scores complete")

## Per-impression AUC

The headline metric is AUC *within* an impression, matching
`evaluation.auc_impression` in the local package and therefore Q4's harness:
a global AUC over pooled rows would reward separating easy impressions from
hard ones rather than ranking candidates inside one. Same rank-sum formula,
vectorised across impressions here because there are hundreds of thousands
of them.

LightGBM's built-in (global) `auc` is still used for early stopping, where
only the stopping point matters and the cheaper metric is adequate.

In [ ]:
def per_impression_auc(impression_ids, scores, labels) -> float:
    """Mean over impressions of the rank-sum AUC, ties given average rank --
    the same definition as evaluation.auc_impression. Impressions without
    both a click and a non-click are skipped (AUC undefined), exactly as the
    local implementation raises on them."""
    df = pd.DataFrame({"i": impression_ids, "s": scores, "y": np.asarray(labels, dtype=np.float64)})
    # average-rank within each impression, ascending by score
    df["r"] = df.groupby("i")["s"].rank(method="average")
    g = df.groupby("i")
    n = g.size()
    n_pos = g["y"].sum()
    rank_sum_pos = g.apply(lambda d: d.loc[d["y"] > 0, "r"].sum(), include_groups=False)
    n_neg = n - n_pos
    usable = (n_pos > 0) & (n_neg > 0)
    auc = (rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc[usable].mean())


def test_per_impression_auc():
    # perfect, inverted and all-tied rankings, hand-computed
    assert per_impression_auc(["a", "a"], [0.9, 0.1], [1, 0]) == 1.0
    assert per_impression_auc(["a", "a"], [0.1, 0.9], [1, 0]) == 0.0
    assert per_impression_auc(["a"] * 3, [0.5, 0.5, 0.5], [1, 0, 1]) == 0.5
    # mean over two impressions, one perfect one inverted
    assert per_impression_auc(["a", "a", "b", "b"], [0.9, 0.1, 0.1, 0.9], [1, 0, 1, 0]) == 0.5
    # a single-class impression is skipped, not counted as 0 or 1
    assert per_impression_auc(["a", "a", "b", "b"], [0.9, 0.1, 0.9, 0.1], [1, 0, 1, 1]) == 1.0


test_per_impression_auc()
print("ok: per-impression AUC matches hand-computed values and skips single-class impressions")

## Learning curve — is 400,000 impressions enough?

SPEC.md A2 Q2 #3 caps scoring at 400,000 train impressions because each one
costs a BM25 query (~4h to score everything versus ~45min sampled). This
measures whether that cap cost anything: the same configuration is fit on
nested 100k/200k/400k subsamples and scored on the *same* held-out val set.
A flat tail means more scored impressions would not have helped.

In [ ]:
def split_frames(df):
    tr = df[df["split"] == "train"]
    va = df[df["split"] == "val"]
    return tr, va


def subsample_impressions(tr, n_impressions, seed=SEED):
    """Nested subsamples: the 100k set is a subset of the 200k set, which is a
    subset of the 400k set, so successive points differ only by added data
    and the curve is not confounded by drawing disjoint samples."""
    ids = np.sort(tr["impression_id"].unique())
    if len(ids) <= n_impressions:
        return tr, len(ids)
    rng = np.random.default_rng(seed)
    keep = set(rng.permutation(ids)[:n_impressions].tolist())
    return tr[tr["impression_id"].isin(keep)], n_impressions


def fit(tr, va, params=PARAMS, rounds=NUM_BOOST_ROUND):
    dtrain = lgb.Dataset(tr[FEATURE_COLUMNS].to_numpy(np.float32), label=tr["clicked"].astype(np.float32),
                         feature_name=FEATURE_COLUMNS, free_raw_data=True)
    dval = lgb.Dataset(va[FEATURE_COLUMNS].to_numpy(np.float32), label=va["clicked"].astype(np.float32),
                       feature_name=FEATURE_COLUMNS, reference=dtrain, free_raw_data=True)
    booster = lgb.train(
        params, dtrain, num_boost_round=rounds, valid_sets=[dval], valid_names=["val"],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )
    return booster


learning_curves = {}
for name, df in frames.items():
    tr, va = split_frames(df)
    va_scores_input = va[FEATURE_COLUMNS].to_numpy(np.float32)
    rows = []
    for n_impr in LEARNING_CURVE_IMPRESSIONS:
        sub, used = subsample_impressions(tr, n_impr)
        t0 = time.perf_counter()
        booster = fit(sub, va)
        preds = booster.predict(va_scores_input, num_iteration=booster.best_iteration)
        auc = per_impression_auc(va["impression_id"].to_numpy(), preds, va["clicked"].to_numpy())
        rows.append({
            "impressions": used,
            "rows": len(sub),
            "best_iteration": booster.best_iteration,
            "val_auc_per_impression": auc,
            "fit_seconds": round(time.perf_counter() - t0, 1),
        })
        print(f"{name}: {used:>7,} impressions ({len(sub):>10,} rows) -> val AUC {auc:.5f}  ({rows[-1]['fit_seconds']}s)")
        del booster, sub
    learning_curves[name] = rows

learning_curves

In [ ]:
import matplotlib.pyplot as plt

BG = "#121212"
fig, ax = plt.subplots(figsize=(7, 4.2), facecolor=BG)
ax.set_facecolor(BG)
colors = {"ebnerd_large": "#4FC3F7", "mind_large": "#FFB74D"}
for name, rows in learning_curves.items():
    xs = [r["impressions"] for r in rows]
    ys = [r["val_auc_per_impression"] for r in rows]
    ax.plot(xs, ys, marker="o", label=name, color=colors.get(name, "#B0BEC5"), linewidth=2)
    for x, y in zip(xs, ys):
        ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 8),
                    ha="center", color="#E0E0E0", fontsize=8)
ax.set_xlabel("training impressions", color="#E0E0E0")
ax.set_ylabel("val AUC (per impression)", color="#E0E0E0")
ax.set_title("Re-ranker learning curve: does the 400k scoring cap cost accuracy?", color="#FFFFFF")
ax.tick_params(colors="#B0BEC5")
for spine in ax.spines.values():
    spine.set_color("#37474F")
ax.grid(True, color="#2A2A2A", linestyle="--", linewidth=0.7)
legend = ax.legend(facecolor=BG, edgecolor="#37474F", labelcolor="#E0E0E0")
fig.tight_layout()
fig.savefig("/kaggle/working/reranker_learning_curve.png", dpi=150, facecolor=BG)
plt.show()

## Final model per dataset

Refit on the full sampled train split with early stopping on val, then save
the booster plus the metadata the serving adapter needs -- above all the
feature order, which it asserts against.

In [ ]:
models, metadata = {}, {}
for name, df in frames.items():
    tr, va = split_frames(df)
    t0 = time.perf_counter()
    booster = fit(tr, va)
    preds = booster.predict(va[FEATURE_COLUMNS].to_numpy(np.float32), num_iteration=booster.best_iteration)
    auc = per_impression_auc(va["impression_id"].to_numpy(), preds, va["clicked"].to_numpy())

    gain = booster.feature_importance(importance_type="gain")
    importance = sorted(zip(FEATURE_COLUMNS, (gain / gain.sum()).tolist()), key=lambda kv: -kv[1])

    model_path = f"/kaggle/working/reranker_model_{name}.txt"
    booster.save_model(model_path, num_iteration=booster.best_iteration)
    models[name] = booster
    metadata[name] = {
        "schema_version": 1,
        "dataset": name,
        "feature_columns": FEATURE_COLUMNS,   # ORDER MATTERS -- asserted at serving time
        "params": {k: v for k, v in PARAMS.items() if k != "num_threads"},
        "num_boost_round": NUM_BOOST_ROUND,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "best_iteration": booster.best_iteration,
        "train_rows": int(len(tr)),
        "train_impressions": int(tr["impression_id"].nunique()),
        "val_rows": int(len(va)),
        "val_impressions": int(va["impression_id"].nunique()),
        "train_positive_rate": float(tr["clicked"].mean()),
        "val_auc_per_impression": auc,
        "feature_importance_gain": importance,
        "learning_curve": learning_curves[name],
        "fit_seconds": round(time.perf_counter() - t0, 1),
    }
    with open(f"/kaggle/working/reranker_metadata_{name}.json", "w") as f:
        json.dump(metadata[name], f, indent=2)

    print(f"\n{name}: best_iteration={booster.best_iteration}, val AUC (per impression)={auc:.5f}")
    print("   top features by gain:")
    for feat, share in importance[:8]:
        print(f"      {feat:<32} {share:6.2%}")

In [ ]:
def test_saved_models():
    for name in frames:
        model_path = f"/kaggle/working/reranker_model_{name}.txt"
        meta_path = f"/kaggle/working/reranker_metadata_{name}.json"
        assert os.path.exists(model_path) and os.path.exists(meta_path)

        # a reloaded booster must reproduce the in-memory one exactly,
        # otherwise the model served locally is not the model measured here
        reloaded = lgb.Booster(model_file=model_path)
        va = frames[name][frames[name]["split"] == "val"].head(5000)
        X = va[FEATURE_COLUMNS].to_numpy(np.float32)
        a = models[name].predict(X, num_iteration=models[name].best_iteration)
        b = reloaded.predict(X)
        assert np.allclose(a, b, rtol=1e-9, atol=1e-9), "saved model diverges from the trained one"

        meta = json.load(open(meta_path))
        assert meta["feature_columns"] == FEATURE_COLUMNS
        assert reloaded.num_feature() == len(FEATURE_COLUMNS)
        # a re-ranker that beats neither Stage-1 signal on its own would be
        # pointless; AUC must at least be better than chance
        assert meta["val_auc_per_impression"] > 0.5, meta["val_auc_per_impression"]

        # features absent for this dataset must carry no gain: MIND's five
        # all-NaN columns should never produce a split
        all_nan = [c for c in FEATURE_COLUMNS if frames[name][c].isna().all()]
        gains = dict(meta["feature_importance_gain"])
        for c in all_nan:
            assert gains[c] == 0.0, f"{name}: all-NaN feature {c} got gain {gains[c]}"


test_saved_models()
print("ok: saved boosters reload and reproduce their predictions, metadata records the feature order, AUC beats chance, and absent features carry zero gain")

# Manual Verification Complete

Download from `/kaggle/working/`: `reranker_model_{dataset}.txt`,
`reranker_metadata_{dataset}.json`, `reranker_learning_curve.png`. Place the
model and metadata in `data/processed/{dataset}/`, then wire the `reranker`
adapter into `src/evaluation_harness.ipynb`.